In [1]:
import os
import pandas as pd

In [5]:
datasets = {}
for file in os.listdir('.'):
    if file.endswith('.log'):
        datasets[file.split('.')[0]] = pd.read_json(file, lines=True)

conn = datasets['conn']
conn

# 1. Exploration du dataset

On commmence par faire quelques statistiques sur les connexions réseau :

In [15]:
conn["id.orig_h"].value_counts().head(100)

id.orig_h
194.61.24.102                29313
10.42.85.115                  2539
10.42.85.10                    737
fe80::2dcf:e660:be73:d220      116
fe80::a926:8915:319b:2238       18
::                               3
Name: count, dtype: int64

In [10]:
conn["id.resp_h"].value_counts().head(20)

id.resp_h
10.42.85.10       30259
192.168.45.1        584
10.90.90.90         355
ff02::1:3            75
224.0.0.252          73
208.80.153.240       72
10.42.85.255         63
72.21.91.29          49
ff02::1:2            47
204.79.197.203       47
208.80.153.224       45
204.79.197.200       43
40.91.125.0          40
70.37.74.6           38
13.107.21.200        30
74.120.184.194       28
52.242.211.89        22
151.101.1.140        20
151.139.128.14       19
74.120.184.204       16
Name: count, dtype: int64

In [14]:
conn["total_bytes"] = conn.get("orig_bytes", 0) + conn.get("resp_bytes", 0)
top_flows_bytes = (
    conn[["id.orig_h", "id.resp_h", "id.resp_p", "proto", "service", "total_bytes", "orig_bytes", "resp_bytes", "duration"]]
    .sort_values("total_bytes", ascending=False)
    .head(20)
)
top_flows_bytes

,id.orig_h,id.resp_h,id.resp_p,proto,service,total_bytes,orig_bytes,resp_bytes,duration
31709,10.42.85.115,104.119.185.124,443,tcp,ssl,37991129.0,1858.0,37989271.0,109.659907
32078,194.61.24.102,10.42.85.10,3389,tcp,ssl,12853439.0,2366216.0,10487223.0,1810.715048
32089,10.42.85.10,10.42.85.115,3389,udp,rdpeudp,6448171.0,73592.0,6374579.0,950.971623
1698,10.42.85.115,104.18.12.165,443,tcp,ssl,4664235.0,2328.0,4661907.0,126.184417
1340,10.42.85.115,23.47.193.50,443,tcp,ssl,4596703.0,4575.0,4592128.0,33.567848
1333,10.42.85.115,143.204.131.79,443,tcp,ssl,4289666.0,993.0,4288673.0,29.046927
1250,10.42.85.115,151.101.1.67,443,tcp,ssl,4101555.0,10810.0,4090745.0,54.299074
31935,10.42.85.115,23.47.52.14,443,tcp,ssl,3130065.0,2387.0,3127678.0,41.379130
155,10.42.85.115,23.47.48.60,443,tcp,ssl,3050041.0,2387.0,3047654.0,2.925606
32335,10.42.85.115,23.47.52.90,443,tcp,ssl,2645142.0,1030.0,2644112.0,1.100630


# 2. Exploration des IPs internes

On va maintenant explorer les IP internes à notre système pour obtenir plus d'informations sur celui-ci.

Les stats semblent nous montrer qu'on a deux IP internes : 
- 10.42.85.115
- 10.42.85.10 
On va tracer les flux entre ces deux IP pour tenter d'identifier laquelle est quoi.

In [21]:
internal_ips = [
    '10.42.85.115',
    '10.42.85.10'
]
filtered = conn[
    conn["id.orig_h"].isin(internal_ips) &
    conn["id.resp_h"].isin(internal_ips)
]
filtered

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,ip_proto,total_bytes
34,1.600466e+09,CZuQXQ1fVv0we0Cbh9,10.42.85.115,49672,10.42.85.10,88,tcp,krb_tcp,0.002332,236.0,...,True,True,0,ShADdFar,4,408,4,380,6,444.0
35,1.600466e+09,CamdIw1wHdUlXj7JMb,10.42.85.115,49673,10.42.85.10,88,tcp,krb_tcp,0.001230,316.0,...,True,True,0,ShADdFar,5,528,5,1791,6,1895.0
36,1.600466e+09,Cif42d32gIBNGXc8Th,10.42.85.115,49677,10.42.85.10,88,tcp,krb_tcp,0.000928,236.0,...,True,True,0,ShADdFar,4,408,4,380,6,444.0
37,1.600466e+09,Cb59eF4Rse2m83OdB6,10.42.85.115,49676,10.42.85.10,88,tcp,krb_tcp,0.001796,1641.0,...,True,True,0,ShADadFr,6,1893,6,1867,6,3256.0
38,1.600466e+09,CVhYzn44Sk1E2vWXj,10.42.85.115,49671,10.42.85.10,389,tcp,ldap_tcp,0.040866,2238.0,...,True,True,0,ShADdar,7,2530,6,3076,6,5062.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32714,1.600494e+09,CN5Eq63zl6ja4IWiSj,10.42.85.115,51193,10.42.85.10,135,tcp,dce_rpc,10.714642,328.0,...,True,True,0,ShADdFaf,7,620,5,492,6,608.0
32715,1.600494e+09,CkUGPH1mBEIuE4HULj,10.42.85.115,52721,10.42.85.10,389,udp,ldap_udp,0.000335,258.0,...,True,True,0,Dd,1,286,1,206,17,436.0
32718,1.600494e+09,CdHH9u3UO47Kt8VvT,10.42.85.115,59457,10.42.85.10,53,udp,dns,0.001635,47.0,...,True,True,0,Dd,1,75,1,193,17,212.0
32721,1.600494e+09,CsnF201Mb2asuM8a42,10.42.85.115,123,10.42.85.10,123,udp,NaN,0.000935,120.0,...,True,True,0,Dd,1,148,1,148,17,240.0


Maintenant qu'on a identifié les flux on peut aggréger par ip_src, ip_dest, port_dest

In [22]:
agg = (
  filtered.groupby(["id.orig_h", "id.resp_h", "id.resp_p"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
agg

,id.orig_h,id.resp_h,id.resp_p,count
1,10.42.85.115,10.42.85.10,53,656
5,10.42.85.115,10.42.85.10,389,128
2,10.42.85.115,10.42.85.10,88,42
7,10.42.85.115,10.42.85.10,49155,40
4,10.42.85.115,10.42.85.10,135,40
6,10.42.85.115,10.42.85.10,445,23
3,10.42.85.115,10.42.85.10,123,14
8,10.42.85.115,10.42.85.10,49158,3
0,10.42.85.10,10.42.85.115,3389,2


On peut donc remarquer :
- 10.42.85.115 est probablement une workstation
- 10.42.85.10 est probablement un active directory.
On peut dire cela car **10.42.85.115** fait des requêtes à **10.42.85.10** sur le port 88 (kerberos). Qui est un service d'authentification distribué exposé par un AD.

Il est assez étonnant de voir le RDP ( port 3389 ) depuis l'active directory vers le la machine. 
RDP permet d'ouvrir une session graphique à distance sur un ordinateur mais cela n'est pas censé arriver depuis un AD. Pour se renseigner un peu sur le parc, on va regarder de plus près le service kerberos : 

In [27]:
kerberos = datasets['kerberos']
agg = (
  kerberos.groupby(["service", "client"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
agg

,service,client,count
20,krbtgt/C137.LOCAL,DESKTOP-SDN1RPT$/C137.LOCAL,6
19,krbtgt/C137.LOCAL,Administrator/C137.LOCAL,3
26,krbtgt/C137.LOCAL,ricksanchez/C137.LOCAL,3
21,krbtgt/C137.LOCAL,desktop-sdn1rpt$/C137.LOCAL,2
13,desktop-sdn1rpt$,DESKTOP-SDN1RPT$/C137.LOCAL,2
1,LDAP/CITADEL-DC01.C137.local,DESKTOP-SDN1RPT$/C137.LOCAL,1
0,DESKTOP-SDN1RPT$,DESKTOP-SDN1RPT$/C137.LOCAL,1
6,ProtectedStorage/CITADEL-DC01.C137.local,Administrator/C137.LOCAL,1
7,cifs/CITADEL-DC01,Administrator/C137.LOCAL,1
9,cifs/CITADEL-DC01.C137.local,Administrator/C137.LOCAL,1


Comme on peut l'apercevoir dans les requêtes kerberos ci-dessous, nous sommes dans un domaine nommé **C137.LOCAL**.

Les deux machines sont : 
- DESKTOP-SDN1RPT$ soit  10.42.85.115
- CITADEL-DC01 soit 10.42.85.10

On peut identifier plusieurs utilisateurs :
- mortysmith/C137
- ricksanchez/C137
- Administrator/C137.LOCAL


# 3. Reconnaissance périmétrique

Maintenant que nous avons pu faire un peu de reco sur nos données et identifier qui était quoi dans notre jeu de donnée, le premier scénario que nous allons chercher ici est celui du RAT : Remote Access Trojan. On pourrait avoir un malware implanté sur une de nos deux machines qui communiquerait
avec une IP externe pour recevoir ses ordres.

Commençons par regarder nos logs DNS :

In [53]:
dns = datasets['dns']
agg = (dns.groupby(["query", "rejected"]).size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
agg

,query,rejected,count
357,wpad,False,192
15,WPAD,False,72
195,isatap,False,52
287,settings-win.data.microsoft.com,False,45
14,ISATAP,False,39


On va filtrer pour ne garder que ce qui nous intéresse : 

In [55]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

agg[(agg["count"] > 1) & (agg["count"] < 30)][:20]

,query,rejected,count
11,C137,False,29
77,c137,False,24
202,login.live.com,False,21
355,win8.ipv6.microsoft.com,False,21
58,assets.msn.com,False,18
95,checkappexec.microsoft.com,False,17
13,DESKTOP-SDN1RPT,False,16
190,img-s-msn-com.akamaized.net,False,16
75,c.msn.com,False,15
51,arc.msn.com,False,15


On va maintenant regarder du côté des IP externes les plus contactées par des ip internes :

In [64]:
filtered = conn[
    conn["id.orig_h"].isin(internal_ips) &
    ~conn["id.resp_h"].isin(internal_ips)
]

agg = (
  filtered.groupby(["id.orig_h", "id.resp_h", "id.resp_p"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
agg[:10]

,id.orig_h,id.resp_h,id.resp_p,count
2,10.42.85.10,192.168.45.1,53,584
18,10.42.85.115,10.90.90.90,443,354
152,10.42.85.115,208.80.153.240,443,72
8,10.42.85.10,224.0.0.252,5355,67
146,10.42.85.115,204.79.197.203,443,47
258,10.42.85.115,72.21.91.29,80,46
151,10.42.85.115,208.80.153.224,443,45
213,10.42.85.115,40.91.125.0,443,40
144,10.42.85.115,204.79.197.200,443,39
254,10.42.85.115,70.37.74.6,443,38


Dans les IP externes les plus contactées on a donc :
- 192.168.45.1 ( probablement le resolveur DNS local car contacté sur le port 53)
- 10.90.90.90
- 208.80.153.240
Si on regarde par quel nom de domaine ces IP sont résolues :
- 10.90.90.90 : settings-win.data.microsoft.com
- 208.80.153.240 : upload.wikimedia.org

Ces connexions ne sont à priori pas très suspectes, on va essayer de tenter une autre hypothèse : et si une IP externe avait abusé d'un des services de notre entreprise pour rentrer ? Alors il faut chercher les connexions d'IP externes vers nos IP privées : 

In [65]:
filtered = conn[
    ~conn["id.orig_h"].isin(internal_ips) &
    conn["id.resp_h"].isin(internal_ips)
]

agg = (
  filtered.groupby(["id.orig_h", "id.resp_h", "id.resp_p"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
agg[:10]

,id.orig_h,id.resp_h,id.resp_p,count
4,194.61.24.102,10.42.85.10,3389,29309
0,194.61.24.102,10.42.85.10,0,1
1,194.61.24.102,10.42.85.10,14,1
2,194.61.24.102,10.42.85.10,80,1
3,194.61.24.102,10.42.85.10,443,1


Alors cette fois on a quelque chose d'intéressant :

On a l'IP externe **194.61.24.102** qui se connecte sur une IP interne **10.42.85.10** qu'on avait pas encore identifié sur le port 3389 (RDP). Cela est très étonnant. Si on regarde ça de plus près dans les logs rdp :

In [78]:
rdp = datasets['rdp']
rdp["dt_utc"] = pd.to_datetime(rdp["ts"], unit="s", utc=True)
filtered = rdp[rdp['id.resp_h'] == '10.42.85.10']
filtered[['dt_utc', 'id.orig_h', 'id.resp_h', 'id.resp_p', 'cookie']][:10]

,dt_utc,id.orig_h,id.resp_h,id.resp_p,cookie
0,2020-09-19 02:19:26.549144983+00:00,194.61.24.102,10.42.85.10,3389,nmap
1,2020-09-19 02:21:26.112469912+00:00,194.61.24.102,10.42.85.10,3389,Administrator
2,2020-09-19 02:21:26.342463017+00:00,194.61.24.102,10.42.85.10,3389,Administrator
3,2020-09-19 02:21:26.564666033+00:00,194.61.24.102,10.42.85.10,3389,Administrator
4,2020-09-19 02:21:26.786791086+00:00,194.61.24.102,10.42.85.10,3389,Administrator
5,2020-09-19 02:21:27.001408100+00:00,194.61.24.102,10.42.85.10,3389,Administrator
6,2020-09-19 02:21:27.224504948+00:00,194.61.24.102,10.42.85.10,3389,Administrator
7,2020-09-19 02:21:27.437500954+00:00,194.61.24.102,10.42.85.10,3389,Administrator
8,2020-09-19 02:21:27.661672115+00:00,194.61.24.102,10.42.85.10,3389,Administrator
9,2020-09-19 02:21:27.887578964+00:00,194.61.24.102,10.42.85.10,3389,Administrator


Ci-dessus on peut remarquer que la première connexion RDP **194.61.24.102** -> **10.42.85.10** a eu lieu le **2020-09-19 à 02:19:26**.

Le champ cookie nous indique que l'outil **nmap** a probablement été utilisé pour faire du repérage sur la machine. 
Deux minutes plus tard, on a des tentatives de connexions qui commencent à être visibles avec le compte administrateur toujours depuis la même IP.
Ces connexions ressemblent forcement à un bruteforce étant donné le rapprochement entre chaque connexion. 

On peut même utiliser la visualition pour observer ces différentes connexions RDP dans notre dataset : 

In [82]:
from msticpy.vis import mp_pandas_plot

filtered.mp_plot.timeline(time_column="dt_utc");

Loading BokehJS ...

A priori zeek ne nous donne pas tellement plus de détails sur la connexion.

In [83]:
ssl = datasets['ssl']
ssl[:10]

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,version,cipher,curve,server_name,resumed,next_protocol,established,ssl_history,cert_chain_fps,client_cert_chain_fps,sni_matches_cert,validation_status
0,1.600466e+09,CEuv8I265VfWODWJh8,10.42.85.115,49688,10.90.90.90,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,settings-win.data.microsoft.com,False,http/1.1,True,CsxknGIti,[4f676a3036d9b59d190f018cc6716a7431c62896e2f761187bf251870900c60e],[],0.0,self signed certificate
1,1.600466e+09,CUaqSfXIhjNIElhNi,10.42.85.115,49696,10.90.90.90,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,watson.telemetry.microsoft.com,False,http/1.1,True,CsxknGIti,[4f676a3036d9b59d190f018cc6716a7431c62896e2f761187bf251870900c60e],[],0.0,self signed certificate
2,1.600466e+09,CNwo5d3HbxgRt9VND3,10.42.85.115,49697,10.90.90.90,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,settings-win.data.microsoft.com,False,http/1.1,True,CsxknGIti,[4f676a3036d9b59d190f018cc6716a7431c62896e2f761187bf251870900c60e],[],0.0,self signed certificate
3,1.600466e+09,CZM4Ke4AMpFKwFrli5,10.42.85.115,49698,10.90.90.90,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,settings-win.data.microsoft.com,False,http/1.1,True,CsxknGIti,[4f676a3036d9b59d190f018cc6716a7431c62896e2f761187bf251870900c60e],[],0.0,self signed certificate
4,1.600466e+09,CicMpB34EdKm1Uqfdg,10.42.85.115,49715,52.114.159.23,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,x25519,v20.events.data.microsoft.com,False,NaN,True,CsxknGIi,"[3896d8ced6e19983361bc711b3dcb290ac29cde5a217de2ca49e6e40580450a3, 83688f2aef71386e0936c4b3013b07e8e0c796d8427716dd48b2a63d79509129]",[],1.0,unable to get local issuer certificate
5,1.600466e+09,C6Imcj1Sv3vQGRvCki,10.42.85.115,49719,204.79.197.200,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,www.bing.com,False,h2,True,CsxuknGIti,"[82a746b2a427722191020b8438db6255469786ce89c6d2479337dca654d9f724, 4e107c981b42acbe41c01067e16d44db64814d4193e572317ea04b87c79c475f]",[],1.0,unable to get local issuer certificate
6,1.600466e+09,CunLRz1ydaEetpi7ah,10.42.85.115,49721,104.126.211.229,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp256r1,go.microsoft.com,False,http/1.1,True,CsxuknGIti,"[728b8206150e3a5f41b0b8eff55aad04dce8762aeda06bc9c857b5291680b354, f0ee5914ed94c7252d058b4e39808aee6fa8f62cf0974fb7d6d2a9df16e3a87f]",[],1.0,unable to get local issuer certificate
7,1.600466e+09,CSQO7x3rV5uu4pqFWc,10.42.85.115,49720,104.126.211.229,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp256r1,go.microsoft.com,False,http/1.1,True,CsxuknGIti,"[728b8206150e3a5f41b0b8eff55aad04dce8762aeda06bc9c857b5291680b354, f0ee5914ed94c7252d058b4e39808aee6fa8f62cf0974fb7d6d2a9df16e3a87f]",[],1.0,unable to get local issuer certificate
8,1.600466e+09,C0jwVO2o5AFpCNQUBg,10.42.85.115,49725,70.37.74.6,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,x25519,nav.smartscreen.microsoft.com,False,NaN,True,CsxuknGIi,"[ced9209561756d9ae0b4dc854228b0eefcc025cc91f67f09622a9ea5cba53792, 4ff404f02e2cd00188f15d1c00f4b6d1e38b5a395cf85314eaeba855b6a64b75]",[],1.0,unable to get local issuer certificate
9,1.600466e+09,COclxv4LzLnBlMJRB7,10.42.85.115,49723,204.79.197.203,443,TLSv12,TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384,secp384r1,www.msn.com,False,h2,True,CsxuknGIti,"[a3d27bac78afb4513800d407b691daeeff51913070f4f985260bcca4a40272c6, f0ee5914ed94c7252d058b4e39808aee6fa8f62cf0974fb7d6d2a9df16e3a87f]",[],1.0,unable to get local issuer certificate
